In [5]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

MODEL = 'llama3.2:1b'
openai = OpenAI(base_url="http://localhost:11434/v1",api_key="ollama")

links = fetch_website_links("https://www.kontekindustries.com/")

link_system_prompt = """
You are an expert web analyst helping to build a business brochure.
Your task is to review a list of website links and select exactly 5 URLs that contain the most important information for a brochure.

The 5 links you select MUST cover these core categories:
1. The primary landing page summarizing the core business value and main tagline.
2. Pages detailing who runs the company, its mission, history, or background.
3. Pages displaying what the business sells, features, or solutions provided.
4. Pages showing cost, subscription tiers, plans, or menu options.
5. Pages containing phone numbers, emails, physical addresses, maps, or customer reviews.

RULES:
- DO NOT select blog posts, news articles, privacy policies, terms of service, login/signup pages, or shopping cart pages.
- Prefer top-level, general pages over highly specific sub-pages (e.g., prefer "/services" over "/services/marketing/seo").
- If a category is missing from the provided links, do your best to pick the next most relevant page.

OUTPUT FORMAT
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""


def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.
Ignore all website navigation menus, header links (e.g., 'Home', 'About', 'Login', 'Cart'), footers, and boilerplate text.
Do not add telephone numbers as links.
Focus ONLY on the core business content, articles, or product descriptions found in the main body of the text.
Select all relevant links for a brochure and output the JSON.
Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt




def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

short_content_text = """
You are an expert data extractor helping to build a business brochure. Your task is to read raw text scraped from a webpage and extract only the most important facts.

RULES for Extraction:
1. Extract exactly 3 to 5 key facts that are highly relevant for a business brochure.
2. Focus strictly on the core subject of the page (e.g., if the category is "Pricing", only extract prices and features).
3. Completely ignore navigation menus, cookie notices, generic marketing fluff, and irrelevant links.
4. Output your response ONLY as a simple markdown bulleted list.
5. DO NOT include any conversational filler, introductory sentences, or concluding remarks.
"""




def short_content( url,type):

    my_scraped_text = fetch_website_contents(url)
    if not my_scraped_text:
        my_scraped_text = "No content available from this page."

    user_prompt_prefix = f"""Here is the text extracted by my web scraper for a specific page. Please extract the key facts based on your system instructions.
    PAGE CATEGORY: {type}
    PAGE URL: {url}

    SCRAPED TEXT:
    \"\"\"
    {my_scraped_text}
    \"\"\"
    """
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": short_content_text},
            {"role": "user", "content": user_prompt_prefix}
        ]
    )
    result = response.choices[0].message.content
    return result


def fetch_page_and_all_relevant_links(url):

    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:with content"
    for link in relevant_links['links']:
        result += f"\n\nLink: {link['type']}\n"
        result += short_content(url , link['type'])
    return result

brochure_system_prompt = """
You are an expert marketing copywriter and graphic designer creating a professional business brochure. Your task is to assemble the final brochure using the provided company name, links, and page summaries.

CRITICAL RULES FOR HANDLING MISSING DATA:
- Some page summaries or links may be marked as "nil", "None", or be completely empty.
- DO NOT crash, complain, or output error messages if data is missing.
- If a section (like Pricing or About Us) is "nil", gracefully omit that section or generate a brief, professional placeholder statement based on the company name and available context.
- Never hallucinate fake addresses, phone numbers, or exact pricing if they return as nil; instead, use general placeholder text (e.g., "Contact us for custom pricing").

OUTPUT FORMAT:
Structure the brochure clearly into panels:
You are a professional layout copywriter. Your job is to format the provided company data into a clean, structured 3-panel business brochure.

CRITICAL FORMATTING RULES:
1. You must use exact markdown headings for each panel as shown below.
2. Do not write a long essay. Use short paragraphs and bullet points.
3. If any information is missing or marked as nil, write a professional placeholder instead of crashing.

YOU MUST STRICTLY FOLLOW THIS EXACT STRUCTURE:

# PANEL 1: FRONT COVER
- **Company Name:** [Insert Name]
- **Main Tagline:** [Catchy 1 sentence value proposition]
- **Call to Action:** [e.g., Visit us today!]

# PANEL 2: INSIDE PANELS
## About Us
- [1-2 bullet points about the company background]

## Core Services & Products
- [Bullet point 1]
- [Bullet point 2]
- [Bullet point 3]

## Pricing & Packages
- [Pricing details or placeholder]

# PANEL 3: BACK COVER
- **Contact Info:** [Email, phone, or location if available]
- **Website:** [Main URL]
- **Final Message:** [Short closing slogan]
"""


def get_brochure_user_prompt(company_name, url):
    scraped_content = fetch_page_and_all_relevant_links(url)

    # SAFETY FIX: If the scraper returns None or empty, default to a safe string
    if not scraped_content:
        scraped_content = "No content could be extracted from these links."

    user_prompt = f"""
    You are looking at a company called: {company_name}
    Here are the contents of its landing page and other relevant pages;
    use this information to build a short brochure of the company in markdown without code blocks.\n\n
    {scraped_content}
    """
    # user_prompt = user_prompt[:1_000] # Truncate if more than 5,000 characters
    display(Markdown(user_prompt))
    return user_prompt


def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="llama3.2:1b",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

API key looks good so far


In [6]:
create_brochure('Telus Digital','https://www.telusdigital.com/')

Selecting relevant links for https://www.telusdigital.com/ by calling llama3.2:1b
Found 2 relevant links



    You are looking at a company called: Telus Digital
    Here are the contents of its landing page and other relevant pages;
    use this information to build a short brochure of the company in markdown without code blocks.


    ## Landing Page:with content

Link: about page
Here are the extracted key facts for the Telus Digital "about" page:

* Telsol Digital provides AI Transformation, Digital Solutions, and Customer Experience solutions.
* The company has multiple offices worldwide, including a focus on customer experience.
* Fuel iX technologies provide a comprehensive end-to-end solution for digital transformation.
* Telsol Digital's services include Trust & Safety, Data Insights & Analytics Services, and Technology Partnerships.

Link: careers page
Here are the key facts extracted from the text in a markdown bulleted list:

* **CX Management**: Humanitarian approach to building customer experiences through AI and Machine Learning.
* **Trust & Safety**: Security measures and solutions offered by Telus Digital to protect customers and data.
* **Fuel iX Product Lineup**: A portfolio of products including Fuel iX Fortify, Agent Trainer, Copilots, Assist, Platform, Technology Partners, Overview, and more.
    

**Telus Digital Business Brochure**

### About Us
We are Telus Digital, a global leader in delivering transformative digital solutions that improve customer experiences.

* Our mission is centered around providing AI-powered expertise to drive business success through:
  + **AI Transformation**: Unlocking new value through data-driven decision making.
  + **Digital Solutions**: Empowering efficient and effective online engagement.
  + **Customer Experience**: Crafting tailored, omnichannel experiences for all customers.

### Who We Are
With a presence in multiple offices worldwide, we prioritize customer-centricity in everything we do. Our focus includes:

* **Customer Experience Management**: Elevating the way you interact with our digital services and products.
* **Trust & Safety**: Safeguarding data integrity through robust security measures.
* Fuel iX technologies provide a comprehensive end-to-end solution for digital transformation underpinned by fuelX Fortify, Agent Trainer, Copilots, Assistent and Platform.